In [19]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.express as px

# Task 1 · Part 1 — From headlines to a dataset

**Guiding question:** Can Peruvian news headlines be transformed into a reliable news classifier?

The performance of any supervised classifier is ultimately bounded by the quality of the data on which it is trained. Accordingly, this first notebook addresses the data-generation stage of the project: it collects real headlines from a Peruvian newspaper, cleans and normalizes them, and assigns each item to one of four predefined thematic categories. In doing so, it establishes the empirical foundation upon which the subsequent modelling stage rests.

| Parameter | Value |
|---|---|
| **Download date** | 2026-10-18 |
| **Outlet** | Option B — El Comercio (`elcomercio.pe`) |
| **Source of article URLs** | Official section sitemaps: `https://elcomercio.pe/sitemap/web/{section}/?outputType=xml` |
| **Source of article fields** | Metadata of each article page (`og:title`, `og:description`, `article:published_time`) |
| **Sections retrieved** | `politica`, `economia`, `deporte-total`, `tecnologia` |

**Methodological note on the choice of outlet.** La República (Option A) was initially considered. However, a preliminary assessment (see below) revealed that each of its RSS feeds exposes only around 21 items, a volume insufficient to reach the required 90–100 news items per category. Consequently, El Comercio was selected, since its section sitemaps list approximately 100 articles each.

**Ethical considerations.** El Comercio's `robots.txt` explicitly lists these sitemaps for automated access and does not restrict article pages. Furthermore, a pause is introduced between consecutive requests in order to minimize the load imposed on the outlet's servers, in line with responsible web-scraping practices.

In [20]:
import os
import time
from datetime import date

import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.express as px

In [21]:
# =============================================
#  CONFIGURATION
# =============================================

fecha_descarga = date.today().isoformat()

# A descriptive User-Agent tells the outlet who is making the requests and why.
headers = {"User-Agent": "Mozilla/5.0 (Diplomado PUCP - Python Task 1, academic use)"}

# The assignment requires pauses of 1 to 2 seconds between requests.
pausa_segundos = 1.5

# Final source: El Comercio section sitemaps.
secciones = ["politica", "economia", "deporte-total", "tecnologia"]
url_sitemap = "https://elcomercio.pe/sitemap/web/{seccion}/?outputType=xml"

# Kept only for the preliminary assessment of La República (Option A).
url_rss = "https://larepublica.pe/rss/{seccion}.xml"

fecha_descarga

'2026-10-18'

All parameters are centralized in a single configuration cell, which enhances the reproducibility of the notebook: should the collection need to be repeated, only this cell must be modified. In this regard, the descriptive `User-Agent` transparently identifies the academic purpose of the requests, and the 1.5-second pause complies with the assignment's guideline of waiting between one and two seconds per request. Moreover, the `{seccion}` placeholder allows the same URL template to be reused for each of the four sections. Finally, the La República RSS template is retained solely to document the preliminary assessment that motivated the change of outlet.

## 1. Preliminary assessment: La República (Option A)

Before committing to a data source, its capacity to supply the required volume of news was assessed empirically.

In [22]:
# Test a single section before looping over all four.
respuesta = requests.get(url_rss.format(seccion="politica"), headers=headers, timeout=30)
respuesta.status_code

200

A single test request is sent to La República's politics RSS feed. The resulting status code indicates whether access was granted: `200` confirms a successful request, whereas `403` would signal that the outlet blocked it and `429` that requests were being sent too frequently.

In [23]:
# RSS is XML: each news article lives inside an <item> tag.
sopa = BeautifulSoup(respuesta.content, "xml")
items = sopa.find_all("item")
len(items)

21

Since RSS feeds are structured as XML documents, each news article is enclosed in an `<item>` tag, and `find_all("item")` therefore returns every article available in the feed. Although access was granted (status `200`), the feed exposes only 21 items. Extrapolating to the four sections, the RSS channel would yield roughly 84 news items, well below the required 350. The outlet's sitemaps and section pages were also inspected manually, yet none of them exposed more than about 16 items per section. **Accordingly, El Comercio (Option B) was adopted as the data source**, as its section sitemaps list approximately 100 articles each.

## 2. Extraction: collecting article URLs from El Comercio

The extraction proceeds in two stages. First, the section sitemaps are retrieved in order to obtain the list of article URLs; subsequently, each article page is visited to extract its fields. This section addresses the first stage.

In [24]:
# Create the folder for raw backups required by the assignment.
os.makedirs("datos/raw", exist_ok=True)

The assignment requires a raw backup of the collected data in `datos/raw/`. Accordingly, this folder is created beforehand; the argument `exist_ok=True` prevents an error if the folder already exists, so the notebook can be re-run safely.

In [25]:
lista_enlaces = []

for seccion in secciones:
    respuesta = requests.get(url_sitemap.format(seccion=seccion), headers=headers, timeout=30)
    print(seccion, "-> status", respuesta.status_code)

    # Raw backup: the original XML is stored exactly as it was received.
    with open(f"datos/raw/sitemap_{seccion}.xml", "wb") as archivo:
        archivo.write(respuesta.content)

    sopa = BeautifulSoup(respuesta.content, "xml")
    for url in sopa.find_all("url"):
        lista_enlaces.append({
            "seccion_sitemap": seccion,
            "url": url.find("loc").text.strip(),
            "lastmod": url.find("lastmod").text.strip(),
        })

    # Pause between requests to avoid overloading the outlet's servers.
    time.sleep(pausa_segundos)

politica -> status 200
economia -> status 200
deporte-total -> status 200
tecnologia -> status 200


This loop iterates over the four sections. For each one, it downloads the sitemap, stores an unaltered copy in `datos/raw/` for traceability, and extracts the URL (`loc`) and last-modification date (`lastmod`) of every article. Each article is appended to a list as a dictionary, which facilitates its later conversion into a table. Finally, a pause is introduced after each request, in compliance with the scraping guidelines.

In [26]:
enlaces = pd.DataFrame(lista_enlaces)
print(enlaces.shape)
enlaces["seccion_sitemap"].value_counts()

(400, 3)


seccion_sitemap
politica         100
economia         100
deporte-total    100
tecnologia       100
Name: count, dtype: int64

The list of dictionaries is converted into a DataFrame, whose dimensions indicate the total number of URLs collected. In addition, the count per section verifies whether each sitemap provides a sufficient number of articles to reach the target of 90–100 news items per category.

## 3. Extraction: retrieving the fields of each article

In the second stage, each article page is visited in order to extract its headline, summary and publication date from the page metadata. Specifically, the Open Graph tags `og:title` and `og:description`, together with `article:published_time`, provide these fields in a standardized format, which is considerably more stable than relying on the visual layout of the page.

In [27]:
lista_noticias = []

for posicion, fila in enlaces.iterrows():
    estado = None
    html = ""

    # First attempt plus at most one retry, as required by the assignment.
    for intento in range(2):
        try:
            respuesta = requests.get(fila["url"], headers=headers, timeout=30)
            estado = respuesta.status_code
            html = respuesta.text
        except requests.RequestException:
            estado = "connection error"
        if estado == 200:
            break
        time.sleep(pausa_segundos)

    sopa = BeautifulSoup(html, "html.parser")
    titulo = sopa.find("meta", attrs={"property": "og:title"})
    descripcion = sopa.find("meta", attrs={"property": "og:description"})
    publicacion = sopa.find("meta", attrs={"property": "article:published_time"})

    lista_noticias.append({
        "url": fila["url"],
        "seccion_sitemap": fila["seccion_sitemap"],
        "estado": estado,
        "titular": titulo["content"] if titulo else None,
        "bajada": descripcion["content"] if descripcion else None,
        "fecha": publicacion["content"] if publicacion else None,
    })

    # Progress message every 50 articles, since the loop takes several minutes.
    if posicion % 50 == 0:
        print(posicion, "articles processed")

    time.sleep(pausa_segundos)

0 articles processed
50 articles processed
100 articles processed
150 articles processed
200 articles processed
250 articles processed
300 articles processed
350 articles processed


This loop visits each of the 400 URLs. For robustness, every request is attempted at most twice: if the first attempt fails, the loop waits and retries once, after which it records the outcome in the `estado` column rather than interrupting the process. In this way, failed requests are not silently lost but can later be reported as discarded items. Moreover, if a metadata tag is absent, the corresponding field is stored as missing (`None`) instead of raising an error. Finally, the pause after each request keeps the collection within the assignment's ethical guidelines.

In [28]:
noticias_crudas = pd.DataFrame(lista_noticias)

# Raw backup before any cleaning, for traceability.
noticias_crudas.to_csv("datos/raw/noticias_crudas.csv", index=False)

print(noticias_crudas.shape)
noticias_crudas["estado"].value_counts()

(400, 6)


estado
200    400
Name: count, dtype: int64

The collected records are converted into a DataFrame and immediately stored in `datos/raw/` before any transformation is applied, thereby preserving an untouched version of the data. Furthermore, the distribution of the `estado` column summarizes how many requests succeeded (`200`) and how many failed, which constitutes the first input for the report of discarded news items.

## 4. Cleaning and normalization

Raw web data rarely meets the requirements of an analytical dataset. Therefore, this section derives the original section of each article, removes failed requests, duplicates and empty headlines, maps each section onto the four official categories, and documents every discarded item.

In [29]:
def extraer_seccion(url):
    """Return the section path of an El Comercio URL, e.g. 'tecnologia-actualidad'."""
    partes = url.replace("https://elcomercio.pe/", "").strip("/").split("/")
    # The last part is the article slug; the previous ones form the section path.
    return "-".join(partes[:-1])


noticias = noticias_crudas.copy()
noticias["seccion_original"] = noticias["url"].apply(extraer_seccion)
noticias["seccion_principal"] = noticias["url"].str.replace("https://elcomercio.pe/", "").str.split("/").str[0]
noticias["medio"] = "El Comercio"

noticias[["url", "seccion_principal", "seccion_original"]].head()

,url,seccion_principal,seccion_original
0,https://elcomercio.pe/politica/gobierno/jorge-...,politica,politica-gobierno
1,https://elcomercio.pe/politica/alvaro-vargas-l...,politica,politica
2,https://elcomercio.pe/politica/el-senado-tambi...,politica,politica
3,https://elcomercio.pe/politica/el-voto-31-este...,politica,politica
4,https://elcomercio.pe/politica/comision-de-con...,politica,politica


El Comercio does not publish the section as a separate field; however, it is embedded in the structure of each URL. Accordingly, the function `extraer_seccion` removes the domain and the article slug, so that the remaining path becomes `seccion_original` (e.g., `tecnologia/actualidad/` becomes `tecnologia-actualidad`). In addition, `seccion_principal` retains only the first level of the path, which will serve as the basis for the category mapping. Since every article comes from the same outlet, the `medio` column is assigned a constant value.

In [30]:
# Requests that did not return 200 cannot provide valid fields.
fallidas = (noticias["estado"] != 200).sum()
noticias = noticias[noticias["estado"] == 200]

duplicados = noticias.duplicated(subset="url").sum()
noticias = noticias.drop_duplicates(subset="url")

# Missing headlines and headlines with only blank spaces are both treated as empty.
vacios = (noticias["titular"].isna() | (noticias["titular"].str.strip() == "")).sum()
noticias = noticias[noticias["titular"].notna() & (noticias["titular"].str.strip() != "")]

print("Failed requests :", fallidas)
print("Duplicate URLs  :", duplicados)
print("Empty headlines :", vacios)

Failed requests : 0
Duplicate URLs  : 1
Empty headlines : 0


Three cleaning rules are applied sequentially. First, failed requests are excluded, since they cannot provide valid fields. Second, duplicates are removed using the URL as the unique identifier, given that an article may appear more than once across sitemaps. Third, empty headlines are discarded, including those consisting only of blank spaces. The number of items removed at each step is stored, as it will feed the report of discarded news.

In [31]:
# Explicit mapping from El Comercio's main sections to the four official categories.
mapeo_secciones = {
    "politica": "politica",
    "economia": "economia",
    "deporte-total": "deportes",
    "tecnologia": "ciencia_tecnologia",
}

noticias["etiqueta"] = noticias["seccion_principal"].map(mapeo_secciones)

tabla_equivalencias = noticias.copy()
tabla_equivalencias["etiqueta"] = tabla_equivalencias["etiqueta"].fillna("(discarded)")
tabla_equivalencias = (
    tabla_equivalencias.groupby(["seccion_original", "etiqueta"])
    .size()
    .reset_index(name="noticias")
    .sort_values("etiqueta")
)
tabla_equivalencias

,seccion_original,etiqueta,noticias
0,bbc-news-mundo,(discarded),2
11,ecdata,(discarded),4
18,eldominical-ciencia,(discarded),1
19,elecciones,(discarded),1
20,opinion-columnistas,(discarded),6
29,ruedas-tuercas-automotriz,(discarded),2
30,tecnologia-actualidad,ciencia_tecnologia,37
31,tecnologia-ciberseguridad,ciencia_tecnologia,7
32,tecnologia-ciencias,ciencia_tecnologia,10
33,tecnologia-ecologia,ciencia_tecnologia,6


The mapping is defined explicitly through a dictionary that links each main section of El Comercio to one of the four official categories. Consequently, every subsection inherits the category of its parent section (e.g., `tecnologia-inteligencia-artificial` is assigned to `ciencia_tecnologia`). Conversely, articles whose main section is not included in the dictionary, such as items republished from other sections, receive no category and are marked as discarded. The resulting table makes the full section-to-category equivalence transparent, as required by the assignment.

In [32]:
fuera_de_categoria = noticias["etiqueta"].isna().sum()
noticias = noticias[noticias["etiqueta"].notna()]

reporte_descartes = pd.DataFrame({
    "motivo": ["Failed request", "Duplicate URL", "Empty headline", "Outside the four categories"],
    "noticias_descartadas": [fallidas, duplicados, vacios, fuera_de_categoria],
})
reporte_descartes

,motivo,noticias_descartadas
0,Failed request,0
1,Duplicate URL,1
2,Empty headline,0
3,Outside the four categories,16


Following the assignment's instructions, news items that do not fit any of the four categories are discarded rather than assigned to an additional category. The table above consolidates the number of items removed under each criterion, thus providing a transparent account of the cleaning process.

In [33]:
# Keep only the date (AAAA-MM-DD) from the full timestamp.
noticias["fecha"] = noticias["fecha"].str[:10]

columnas_finales = ["titular", "bajada", "medio", "seccion_original", "url", "fecha", "etiqueta"]
noticias = noticias[columnas_finales].reset_index(drop=True)

print(noticias.shape)
noticias["etiqueta"].value_counts()

(383, 7)


etiqueta
economia              99
deportes              98
ciencia_tecnologia    95
politica              91
Name: count, dtype: int64

Finally, the publication timestamp is reduced to the `AAAA-MM-DD` format, and only the seven mandatory columns are retained in the order specified by the assignment. The resulting dimensions and the distribution of categories confirm whether the dataset meets the minimum volume of 350 news items.